[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/07_Final_Summary.ipynb)

# Notebook 07 — Integrated Findings and Tiered Architecture Recommendation

This notebook reads the saved results of Notebooks 03–06, which must have been run first. It produces:
* **Table 19:** the answer to each research question, with its key evidence;
* **Table 20 / Figure 17:** the recommended **tiered review-analytics architecture**, with the expected accuracy, latency and cost of each tier;
* **Figure 18:** a **cascade simulation** that sends only uncertain reviews to the foundation model, showing the accuracy-cost trade-off of routing;
* a plain-text summary of the findings, generated from the numbers, as a starting point for the report.

No model training happens here, so a CPU runtime is enough.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

In [ ]:
import json
import cvi_utils as U
U.set_project(PROJECT_DIR)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
U.plot_style()
RES = U.P("outputs", "results"); TAB = U.P("outputs", "tables")
NAMES = {"logreg": "TF-IDF + LogReg", "ffn": "Feed-forward NN", "cnn": "1D-CNN", "bilstm": "BiLSTM"}

def load_json(name):
    p = f"{RES}/{name}"
    if not os.path.exists(p):
        print(f"Missing {name}: run the corresponding notebook first."); return {}
    return json.load(open(p))

rq1, rq2, rq3, rq4 = (load_json(f) for f in ["rq1_decision.json", "rq2_decision.json", "rq3_decision.json", "rq4_decision.json"])
runs = pd.read_csv(f"{RES}/rq1_runs.csv"); lat = pd.read_csv(f"{RES}/rq1_latency.csv").set_index("model")
shift = pd.read_csv(f"{RES}/rq2_domain_shift.csv").set_index("model")
lc = pd.read_csv(f"{RES}/rq2_learning_curves.csv")
rq4res = pd.read_csv(f"{RES}/rq4_results.csv")
COSTS = json.load(open(f"{RES}/cost_assumptions.json"))

## 1. Table 19 — Answers to the research questions

In [ ]:
final = runs[~runs.key.str.startswith("tune_") & (runs.variant == "raw")]
acc = final.groupby("model").test_accuracy.agg(["mean", "std"])
w = rq1.get("rq1_recommended", "logreg")
lc_mean = lc.groupby(["condition", "n"]).accuracy.mean()
n_max = lc.n.max()
llm_all = rq4res[(rq4res.domain == "all") & rq4res.model.str.startswith("LLM")].sort_values("accuracy", ascending=False)
bert_all = rq4res[(rq4res.domain == "all") & (rq4res.model == "Fine-tuned DistilBERT")].iloc[0]
best_llm = llm_all.iloc[0]
need = rq2.get("labels_needed_in_domain", {}).get("distilbert_imdb_adapted")

table19 = pd.DataFrame([
    ["RQ1 Architecture", f"Most accurate: {NAMES[rq1.get('rq1_most_accurate', w)]} ({100 * acc.loc[rq1.get('rq1_most_accurate', w), 'mean']:.2f}%). "
     f"{NAMES[w]}: {100 * acc.loc[w, 'mean']:.2f}% at {lat.loc[w, 'cpu_ms_per_review']:.2f} ms/review (CPU).",
     f"Use {NAMES[w]} for high-volume triage (least costly within 1 pp of best)."],
    ["RQ2 Transfer", f"DistilBERT in-domain {100 * shift.loc['distilbert', 'imdb_acc']:.2f}%; on Amazon without retraining {100 * shift.loc['distilbert', 'amz_acc']:.2f}% "
     f"(drop {shift.loc['distilbert', 'drop_pp']:.1f} pp; TF-IDF drop {shift.loc['logreg', 'drop_pp']:.1f} pp). "
     f"With {n_max:,} Amazon labels: {100 * lc_mean.loc[('distilbert_imdb_adapted', n_max)]:.2f}%.",
     f"Fine-tune a pre-trained transformer and budget about {need:,} labeled reviews per new category." if need else
     "Fine-tune a pre-trained transformer; the 2 pp recovery target was not reached within 5,000 labels, so budget more labels."],
    ["RQ3 Quality gate", f"Best detector {rq3.get('best_detector')}: AUROC {rq3.get('auroc', np.nan):.3f}, recall {rq3.get('recall', np.nan):.2f}. "
     f"Hybrid gate recall {rq3.get('hybrid_recall', np.nan):.2f}, precision {rq3.get('hybrid_precision', np.nan):.2f}; dashboard error "
     f"{rq3.get('dashboard_error_no_gate_pp', np.nan):.2f} → {rq3.get('dashboard_error_hybrid_pp', np.nan):.2f} pp.",
     "Screen reviews with the autoencoder plus near-duplicate gate before scoring."],
    ["RQ4 Foundation model", f"{best_llm.model}: {100 * best_llm.accuracy:.2f}% at {best_llm.latency_ms:.0f} ms/review vs DistilBERT "
     f"{100 * bert_all.accuracy:.2f}% at {bert_all.latency_ms:.1f} ms/review. Break-even ≈ {rq4.get('break_even_reviews_per_month', np.nan):,.0f} reviews/month.",
     "Use the foundation model for summaries and uncertain cases; self-host the fine-tuned model above the break-even volume."],
], columns=["Research question", "Key evidence", "Decision"])
U.save_table(table19, "table19_rq_answers")
table19.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

## 2. Cascade simulation: routing only uncertain reviews to the foundation model
Using the 2,000 RQ4 reviews, the fine-tuned model scores every review. A review whose probability falls inside an **uncertainty band** around 0.5 is passed to the foundation model instead. Widening the band raises the share of reviews sent to the foundation model, and with it the cost. The simulation shows whether this routing improves accuracy.

In [ ]:
bert = pd.read_csv(f"{RES}/rq4_distilbert_preds.csv")
pred_files = sorted(f for f in os.listdir(RES) if f.startswith("rq4_llm_preds_"))
llm = pd.read_csv(f"{RES}/{pred_files[-1]}")
setting = best_llm.model.replace("LLM ", "").replace("-", "_")
llm = llm[llm.setting == setting][["review_id", "pred"]].rename(columns={"pred": "llm_pred"})
d = bert.merge(llm, on="review_id")
c_bert = bert_all.cost_1k_usd / 1000; c_llm = best_llm.cost_1k_usd / 1000 if pd.notna(best_llm.cost_1k_usd) else np.nan

rows = []
for half in [0, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]:
    route = (d.prob - 0.5).abs() < half
    final_pred = np.where(route, d.llm_pred, d.pred)
    rows.append(dict(band=half, share_to_llm=route.mean(), accuracy=(final_pred == d.label).mean(),
                     cost_1k=1000 * (c_bert + route.mean() * c_llm) if pd.notna(c_llm) else np.nan))
casc = pd.DataFrame(rows); casc.to_csv(f"{RES}/cascade_simulation.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(100 * casc.share_to_llm, 100 * casc.accuracy, "o-", color="#2a7ab9")
for r in casc.iloc[[0, 2, 4, -1]].itertuples():
    ax.annotate(f"band ±{r.band:.2f}", (100 * r.share_to_llm, 100 * r.accuracy), xytext=(5, -12), textcoords="offset points", fontsize=8)
ax.set_xlabel("Reviews routed to the foundation model (%)"); ax.set_ylabel("Accuracy of the cascade (%)")
ax.set_title("Figure 18. Cascade: fine-tuned model first, foundation model for uncertain reviews")
U.save_fig(fig, "fig18_cascade"); plt.show()
best_c = casc.loc[casc.accuracy.idxmax()]
print(f"Fine-tuned only: {100 * casc.accuracy[0]:.2f}% | best cascade: {100 * best_c.accuracy:.2f}% with {100 * best_c.share_to_llm:.1f}% of reviews routed (band ±{best_c.band})")
casc.round(4)

## 3. Table 20 and Figure 17 — Recommended tiered architecture

In [ ]:
amz_ft = lc_mean.loc[("distilbert_imdb_adapted", n_max)]
tier1_imdb = acc.loc[w, "mean"]
tier1_amz = lc_mean.loc[("logreg_scratch", n_max)]
table20 = pd.DataFrame([
    ["0. Quality gate", f"{rq3.get('best_detector')} + near-duplicate rule (RQ3)",
     f"Recall {rq3.get('hybrid_recall', np.nan):.2f}, precision {rq3.get('hybrid_precision', np.nan):.2f}; removes {100 * rq3.get('hybrid_normal_removed', np.nan):.1f}% of genuine reviews",
     f"{rq3.get('gate_cpu_ms_per_review', np.nan):.2f} ms (CPU)", f"{rq3.get('gate_cpu_ms_per_review', np.nan) / 3600 * COSTS['cpu_usd_per_hour']:.6f}",
     "All incoming reviews"],
    ["1. Bulk triage", f"{NAMES[w]} (RQ1)", f"{100 * tier1_imdb:.1f}% movies; {100 * tier1_amz:.1f}% products when trained on {n_max:,} product labels",
     f"{lat.loc[w, 'cpu_ms_per_review']:.2f} ms (CPU)", f"{lat.loc[w, 'cost_1k_cpu_usd']:.6f}", "Dashboards, trend monitoring at high volume"],
    ["2. Accurate scoring", f"DistilBERT, domain-adapted (RQ2)", f"{100 * shift.loc['distilbert', 'imdb_acc']:.1f}% movies; {100 * amz_ft:.1f}% products ({n_max:,} labels)",
     f"{bert_all.latency_ms:.1f} ms (GPU)", f"{bert_all.cost_1k_usd:.6f}", "Category launches, decisions needing higher accuracy"],
    ["3. Foundation model", f"{rq4.get('model', 'LLM')} — {best_llm.model} + RAG (RQ4)",
     f"{100 * best_llm.accuracy:.1f}% classification; cited theme summaries; cascade best {100 * best_c.accuracy:.1f}%",
     f"{best_llm.latency_ms:.0f} ms", "n/a" if pd.isna(best_llm.cost_1k_usd) else f"{best_llm.cost_1k_usd:.4f}",
     f"Product summaries; uncertain reviews (~{100 * best_c.share_to_llm:.0f}% routed)"],
], columns=["Tier", "Model", "Expected accuracy / quality", "Latency per review", "Cost per 1k reviews (USD)", "Used for"])
U.save_table(table20, "table20_tiered_architecture")
table20.style.set_properties(**{"text-align": "left"})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2)); ax.set_xlim(0, 12); ax.set_ylim(0, 4.2); ax.axis("off")
def box(x, y, w, h, title, body, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.15", fc=color, ec="0.3", lw=1))
    ax.text(x + w / 2, y + h - 0.28, title, ha="center", va="top", fontsize=9.5, fontweight="bold")
    ax.text(x + w / 2, y + h - 0.68, body, ha="center", va="top", fontsize=8, wrap=True)
def arrow(x1, y1, x2, y2, label="", rad=0.0):
    ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="-|>", color="0.25", lw=1.2, connectionstyle=f"arc3,rad={rad}"))
    if label: ax.text(x1 + 0.4 * (x2 - x1), y1 + 0.4 * (y2 - y1), label, ha="center", va="center", fontsize=7.5, color="0.25",
                      bbox=dict(fc="white", ec="none", pad=1.5))

box(0.1, 1.4, 1.7, 1.4, "Incoming\nreviews", "streaming &\ne-commerce", "#eef1f5")
box(2.3, 1.4, 2.1, 1.4, "Tier 0: Quality gate", f"Autoencoder +\nnear-duplicate rule\nrecall {rq3.get('hybrid_recall', np.nan):.2f}", "#fbe3e6")
box(5.0, 2.45, 2.2, 1.5, "Tier 1: Bulk triage", f"{NAMES[w]}\n{100 * tier1_imdb:.1f}% · {lat.loc[w, 'cpu_ms_per_review']:.2f} ms", "#e3ecf6")
box(5.0, 0.25, 2.2, 1.5, "Tier 2: Accurate scoring", f"DistilBERT (adapted)\n{100 * amz_ft:.1f}% products\n{bert_all.latency_ms:.1f} ms", "#e3f3e5")
box(7.9, 1.35, 2.2, 1.5, "Tier 3: Foundation model", f"{best_llm.model}\nuncertain reviews\n+ RAG summaries", "#f6ead9")
box(10.6, 1.35, 1.3, 1.5, "Insights", "dashboards,\ntheme\nsummaries", "#eef1f5")
arrow(1.8, 2.1, 2.3, 2.1); arrow(4.4, 2.3, 5.0, 3.1, "high volume"); arrow(4.4, 1.9, 5.0, 1.0, "new category")
ax.text(3.35, 1.15, "flagged → review queue", ha="center", fontsize=7.5, color="#d1495b")
arrow(7.2, 3.1, 7.9, 2.4, f"|p − 0.5| < {best_c.band:.2f}"); arrow(7.2, 1.0, 7.9, 1.8, "uncertain")
arrow(7.2, 3.75, 11.25, 2.9, "confident", rad=-0.25); arrow(10.1, 2.1, 10.6, 2.1)
ax.set_title("Figure 17. Recommended tiered review-analytics architecture", fontweight="bold")
U.save_fig(fig, "fig17_architecture"); plt.show()

## 4. Findings summary (generated from the results; edit before pasting into the report)

In [ ]:
lines = [
 f"RQ1. On the official IMDb test set, {NAMES[rq1.get('rq1_most_accurate', w)]} was the most accurate model "
 f"({100 * acc.loc[rq1.get('rq1_most_accurate', w), 'mean']:.2f}%). The least costly model within one point of the best was {NAMES[w]} "
 f"({100 * acc.loc[w, 'mean']:.2f}%, {lat.loc[w, 'cpu_ms_per_review']:.2f} ms per review on a 2-thread CPU).",
 f"RQ2. Fine-tuned DistilBERT reached {100 * shift.loc['distilbert', 'imdb_acc']:.2f}% on IMDb. Applied to Amazon Electronics without retraining, "
 f"accuracy fell by {shift.loc['distilbert', 'drop_pp']:.1f} points (TF-IDF baseline: {shift.loc['logreg', 'drop_pp']:.1f} points). "
 + (f"About {need:,} labeled product reviews restored accuracy to within 2 points of in-domain performance." if need else
    f"Even {n_max:,} labeled product reviews did not restore accuracy to within 2 points of in-domain performance."),
 f"RQ3. The {rq3.get('best_detector')} detector reached AUROC {rq3.get('auroc', np.nan):.3f}. Combined with a near-duplicate rule, it caught "
 f"{100 * rq3.get('hybrid_recall', np.nan):.0f}% of injected anomalies with {100 * rq3.get('hybrid_precision', np.nan):.0f}% precision and reduced the "
 f"dashboard positive-share error from {rq3.get('dashboard_error_no_gate_pp', np.nan):.2f} to {rq3.get('dashboard_error_hybrid_pp', np.nan):.2f} points.",
 f"RQ4. {rq4.get('model', 'The foundation model')} ({best_llm.model}) reached {100 * best_llm.accuracy:.2f}% against {100 * bert_all.accuracy:.2f}% for "
 f"fine-tuned DistilBERT, at {best_llm.latency_ms:.0f} ms versus {bert_all.latency_ms:.1f} ms per review. Self-hosting becomes cheaper above about "
 f"{rq4.get('break_even_reviews_per_month', np.nan):,.0f} reviews per month. Routing only uncertain reviews ({100 * best_c.share_to_llm:.0f}%) "
 f"to the foundation model gave {100 * best_c.accuracy:.2f}% accuracy.",
]
summary = "\n\n".join(lines)
open(U.P("outputs", "findings_summary.txt"), "w").write(summary)
print(summary)

## 5. Package the outputs
This creates a zip file of all tables, figures and result files, so they can be attached to the report or uploaded to GitHub. Model weights and data are excluded.

In [ ]:
import shutil
zip_path = shutil.make_archive(U.P("customer_voice_intelligence_outputs"), "zip", U.P("outputs"))
print("Created", zip_path)
if IN_COLAB:
    from google.colab import files; files.download(zip_path)